# Optimise Models 
Sampling


## 1. Import Libraries

In [1]:
import os
import pandas as pd

from sklearn.model_selection import StratifiedKFold

from config_optimisation import (
    RANDOM_STATE,
    N_SPLITS,
    AT_RISK_RULES,
    COMBINED_AT_RISK_RULES,
    GROUP_COUNT_COLS,
    CONFUSION_COUNT_COLS,
    UNDERSAMPLING_METHOD,
    UNDERSAMPLING_COMBINED_METHOD,
    NON_RISK_PER_RISK_VALUES,
)
from optimisation_helpers import (
    load_model_data,
    make_output_dir,
    get_at_risk_groups,
    count_classes,
    get_skip_reason,
    fit_and_predict,
    evaluate_in_subgroups,
    save_with_sdc,
    plot_confusion_matrices,
    get_at_risk_mask
) 

## 2. Settings

In [2]:
OPTIMISATION_METHOD = UNDERSAMPLING_METHOD
OPTIMISATION_DIR = make_output_dir(OPTIMISATION_METHOD)


## 3. Load Data

In [3]:
data = load_model_data()

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

train_subgroups = data["train_subgroups"]
test_subgroups = data["test_subgroups"]

print("Training data:", X_train.shape)
print("Test data:", X_test.shape)
print("At-risk groups:", AT_RISK_RULES)

Training data: (4000, 87)
Test data: (1000, 87)
At-risk groups: {'age_band': ['75-79', '80-84', '85-89', '90-94', '95-99', '100+'], 'sex': ['female'], 'ethnicity_cat': ['Black', 'Asian'], 'imd_quintile': ['1 (most deprived)', '2'], 'rural_urban': [5, 6, 7, 8], 'mltc_count': {'min': 2}, 'migrant': [1], 'learndis': [1], 'non_english_speaking': [1], 'smi': [1], 'substance_abuse': [1], 'homeless': [1], 'housebound': [1]}


## 4. Cross Validation

In [4]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

## 5. Sample, train and evaluate for each at-risk subgroup

In [ ]:
performance_rows = []
subgroup_rows = []
skipped_rows = []

for group in get_at_risk_groups(
    test_subgroups,
    train_subgroups=train_subgroups,
):

    if not group["test_mask"].any():
        continue

    risk_mask = group["train_mask"]

    risk_data = X_train.loc[risk_mask]
    non_risk_data = X_train.loc[~risk_mask]

    for non_risk_per_risk in NON_RISK_PER_RISK_VALUES:

        sampling_info = {
            "sampling_variable": group["subgroup_variable"],
            "sampling_level": group["subgroup_level"],
            "level_number": group["level_number"],
            "non_risk_per_risk": non_risk_per_risk,
        }

        if risk_data.empty:
            skipped_rows.append({
                **sampling_info,
                "reason": "No training patients in this subgroup",
            })
            continue

        n_non_risk = min(
            len(non_risk_data),
            int(len(risk_data) * non_risk_per_risk),
        )

        if n_non_risk == len(non_risk_data):
            skipped_rows.append({
                **sampling_info,
                "reason": "No patients would be removed",
            })
            continue

        sampled_non_risk = non_risk_data.sample(
            n=n_non_risk,
            random_state=RANDOM_STATE,
        )

        selected_rows = risk_data.index.append(sampled_non_risk.index)

        X_train_sampled = X_train.loc[selected_rows].copy()
        y_train_sampled = y_train.loc[selected_rows].copy()

        counts = count_classes(
            y_train=y_train_sampled,
            y_test=y_test,
        )

        reason = get_skip_reason(counts)

        if reason is not None:
            skipped_rows.append({
                **sampling_info,
                "reason": reason,
            })
            continue

        print(
            f"Sampling for: "
            f"{group['subgroup_variable']} = {group['subgroup_level']}, "
            f"ratio = {non_risk_per_risk}"
        )
        print("Training patients before sampling:", len(X_train))
        print("Training patients after sampling:", len(X_train_sampled))
        print("Selected subgroup patients retained:", len(risk_data))
        print("Other patients retained:", len(sampled_non_risk))

        group_dir = os.path.join(
            OPTIMISATION_DIR,
            f"ratio_{non_risk_per_risk}",
            group["subgroup_variable"],
            f"level_{group['level_number']}",
        )

        performance, predictions = fit_and_predict(
            X_train=X_train_sampled,
            y_train=y_train_sampled,
            X_test=X_test,
            y_test=y_test,
            test_ids=data["test_ids"],
            out_dir=group_dir,
            cv=cv,
            # threshold from the original (not sampled) training set
            X_threshold=X_train,
            y_threshold=y_train,
        )

        subgroup_performance = evaluate_in_subgroups(
            prediction_df=predictions,
            test_subgroups=test_subgroups,
        )

        for column, value in counts.items():
            performance[column] = value

        for column, value in sampling_info.items():
            performance[column] = value
            subgroup_performance[column] = value

        save_with_sdc(
            performance,
            os.path.join(group_dir, "test_performance.csv"),
            count_cols=GROUP_COUNT_COLS + CONFUSION_COUNT_COLS,
        )

        save_with_sdc(
            subgroup_performance,
            os.path.join(group_dir, "subgroup_test_performance.csv"),
            count_cols=[
                "n_test",
                "test_events",
                "test_nonevents",
                *CONFUSION_COUNT_COLS,
            ],
        )

        predictions.to_csv(
            os.path.join(group_dir, "test_predictions.csv.gz"),
            index=False,
            compression="gzip",
        )

        performance_rows.append(performance)
        subgroup_rows.append(subgroup_performance)


Sampling for: age_band = 75-79, 80-84, 85-89, 90-94, 95-99, 100+, ratio = 0.5
Training patients before sampling: 4000
Training patients after sampling: 3172
Selected subgroup patients retained: 2115
Other patients retained: 1057
Fitting 5 folds for each of 6 candidates, totalling 30 fits
lr: best CV roc_auc = 0.5767
lr: best params = {'model__C': 0.1, 'model__l1_ratio': 0.5, 'model__penalty': 'elasticnet', 'model__solver': 'saga'}
Fitting 5 folds for each of 25 candidates, totalling 125 fits


## 6. Combine and Save 

In [ ]:

skipped_df = pd.DataFrame(
    skipped_rows,
    columns=[
        "sampling_variable",
        "sampling_level",
        "level_number",
        "non_risk_per_risk",
        "reason",
    ],
)

skipped_df.to_csv(
    os.path.join(OPTIMISATION_DIR, "skipped_subgroups.csv"),
    index=False,
)

performance_df = pd.concat(
    performance_rows,
    ignore_index=True,
)

subgroup_performance_df = pd.concat(
    subgroup_rows,
    ignore_index=True,
)

save_with_sdc(
    performance_df,
    os.path.join(OPTIMISATION_DIR, "test_performance.csv"),
    count_cols=GROUP_COUNT_COLS + CONFUSION_COUNT_COLS,
)

save_with_sdc(
    subgroup_performance_df,
    os.path.join(
        OPTIMISATION_DIR,
        "subgroup_test_performance.csv",
    ),
    count_cols=[
        "n_test",
        "test_events",
        "test_nonevents",
        *CONFUSION_COUNT_COLS,
    ],
)

print("Sampling experiments fitted:", len(performance_rows))
print("Sampling experiments skipped:", len(skipped_df))
print("Saved results to:", OPTIMISATION_DIR)

## 7. Check 

In [ ]:
display(
    performance_df[
        [
            "sampling_variable",
            "sampling_level",
            "model",
            "n_train",
            "auc",
            "auc_pr",
            "brier",
            "threshold",
        ]
    ].head(20)
)

display(skipped_df)

## 8. Plot Confusion Matrices

In [ ]:
for non_risk_per_risk, ratio_df in performance_df.groupby("non_risk_per_risk"):

    plot_confusion_matrices(
        ratio_df,
        os.path.join(OPTIMISATION_DIR, f"ratio_{non_risk_per_risk}"),
        f"{OPTIMISATION_METHOD}_ratio_{non_risk_per_risk}",
        group_cols=("sampling_variable", "sampling_level"),
    )


## 9. Combined At-Risk Group Settings

In [ ]:
COMBINED_METHOD = UNDERSAMPLING_COMBINED_METHOD

COMBINED_DIR = make_output_dir(COMBINED_METHOD)

at_risk_mask = get_at_risk_mask(train_subgroups, COMBINED_AT_RISK_RULES)

at_risk_data = X_train.loc[at_risk_mask]
non_at_risk_data = X_train.loc[~at_risk_mask]

print("Combined at-risk variables:", list(COMBINED_AT_RISK_RULES))
print("At-risk training patients:", len(at_risk_data))
print("Other training patients:", len(non_at_risk_data))

## 10. Sample, train and evaluate for the combined at-risk group

In [ ]:
combined_rows = []
combined_subgroup_rows = []
combined_skipped_rows = []

for non_risk_per_risk in NON_RISK_PER_RISK_VALUES:

    sampling_info = {
        "sampling_variable": "combined_at_risk",
        "sampling_level": "any",
        "non_risk_per_risk": non_risk_per_risk,
    }

    n_non_risk = min(
        len(non_at_risk_data),
        int(len(at_risk_data) * non_risk_per_risk),
    )

    if n_non_risk == len(non_at_risk_data):
        combined_skipped_rows.append({
            **sampling_info,
            "reason": "No patients would be removed",
        })
        continue

    sampled_non_risk = non_at_risk_data.sample(
        n=n_non_risk,
        random_state=RANDOM_STATE,
    )

    selected_rows = at_risk_data.index.append(sampled_non_risk.index)

    X_train_sampled = X_train.loc[selected_rows].copy()
    y_train_sampled = y_train.loc[selected_rows].copy()

    counts = count_classes(
        y_train=y_train_sampled,
        y_test=y_test,
    )

    reason = get_skip_reason(counts)

    if reason is not None:
        combined_skipped_rows.append({
            **sampling_info,
            "reason": reason,
        })
        continue

    print(
        f"Sampling for combined at-risk group: "
        f"ratio = {non_risk_per_risk}"
    )
    print("Training patients before sampling:", len(X_train))
    print("Training patients after sampling:", len(X_train_sampled))
    print("Other patients retained:", len(sampled_non_risk))

    run_dir = os.path.join(
        COMBINED_DIR,
        f"ratio_{non_risk_per_risk}",
    )

    performance, predictions = fit_and_predict(
        X_train=X_train_sampled,
        y_train=y_train_sampled,
        X_test=X_test,
        y_test=y_test,
        test_ids=data["test_ids"],
        out_dir=run_dir,
        cv=cv,
        # threshold from the original (not sampled) training set
        X_threshold=X_train,
        y_threshold=y_train,
    )

    subgroup_performance = evaluate_in_subgroups(
        prediction_df=predictions,
        test_subgroups=test_subgroups,
    )

    for column, value in counts.items():
        performance[column] = value

    for column, value in sampling_info.items():
        performance[column] = value
        subgroup_performance[column] = value

    predictions.to_csv(
        os.path.join(run_dir, "test_predictions.csv.gz"),
        index=False,
        compression="gzip",
    )

    combined_rows.append(performance)
    combined_subgroup_rows.append(subgroup_performance)

## 11. Combine and Save

In [ ]:

combined_skipped_df = pd.DataFrame(
    combined_skipped_rows,
    columns=["sampling_variable", "sampling_level", "non_risk_per_risk", "reason"],
)
combined_skipped_df.to_csv(os.path.join(COMBINED_DIR, "skipped_subgroups.csv"), index=False)

combined_df = pd.concat(combined_rows, ignore_index=True)
combined_subgroup_df = pd.concat(combined_subgroup_rows, ignore_index=True)

save_with_sdc(
    combined_df,
    os.path.join(COMBINED_DIR, "test_performance.csv"),
    count_cols=GROUP_COUNT_COLS + CONFUSION_COUNT_COLS,
)

save_with_sdc(
    combined_subgroup_df,
    os.path.join(COMBINED_DIR, "subgroup_test_performance.csv"),
    count_cols=[
        "n_test",
        "test_events",
        "test_nonevents",
        *CONFUSION_COUNT_COLS,
    ],
)

print("Combined sampling experiments fitted:", len(combined_rows))
print("Combined sampling experiments skipped:", len(combined_skipped_df))
print("Saved results to:", COMBINED_DIR)